# Answer generation fine-tuning

Practice LoRA training step by step. The two examples below are illustrative, not a dataset sufficient for reliable improvement. No training is started in this notebook yet.


## 1. Dependencies
Run if needed. After installation, restart the kernel and continue below.


In [ ]:
%pip install datasets peft accelerate


## 2. Hardware
Close other model-heavy notebook kernels before loading Qwen.


In [ ]:
import torch

if torch.cuda.is_available():
    device = "cuda"
    gpu = torch.cuda.get_device_properties(0)
    print("GPU:", gpu.name)
    print("GPU memory:", round(gpu.total_memory / 1024**3, 1), "GB")
elif torch.backends.mps.is_available():
    device = "mps"
    print("Apple Silicon GPU")
else:
    device = "cpu"
    print("CPU only")

print("Device:", device)
print("PyTorch:", torch.__version__)


## 3. Example conversations
Replace illustrative evidence with verified PDF passages before meaningful training.


In [ ]:
training_examples = [
    {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are an exam study assistant. "
                    "Answer using only the supplied evidence. "
                    "Cite factual claims using source IDs such as [S1]. "
                    "If evidence is insufficient, say so."
                ),
            },
            {
                "role": "user",
                "content": (
                    "Question:\n"
                    "What are the disadvantages of cloud computing?\n\n"
                    "Evidence:\n"
                    "[S1] Users may become locked into a cloud vendor's "
                    "offerings. Overall costs may be higher than expected."
                ),
            },
            {
                "role": "assistant",
                "content": (
                    "Cloud computing can create vendor lock-in and "
                    "may lead to higher costs than expected. [S1]"
                ),
            },
        ]
    }
]

print(training_examples[0])


In [ ]:
training_examples.append(
    {
        "messages": [
            {
                "role": "system",
                "content": training_examples[0]["messages"][0]["content"],
            },
            {
                "role": "user",
                "content": (
                    "Question:\n"
                    "How much money will my university save by using cloud computing?\n\n"
                    "Evidence:\n"
                    "[S1] Cloud computing costs depend on usage, "
                    "and overall costs may be higher than expected."
                ),
            },
            {
                "role": "assistant",
                "content": (
                    "The supplied evidence does not provide enough information "
                    "to calculate your university's savings. It only states "
                    "that costs depend on usage and may exceed expectations. [S1]"
                ),
            },
        ]
    }
)

print("Training examples:", len(training_examples))


## 4. Save and load JSONL
Saving preserves an existing examples.jsonl; delete or rename it explicitly if you want to replace it with these practice examples.


In [ ]:
import json
from pathlib import Path

PROJECT_ROOT = Path(
    "/Volumes/code/MyProjects/ML/University Handbook Assistant From Scratch"
)
training_dir = PROJECT_ROOT / "data" / "fine_tuning"
training_dir.mkdir(parents=True, exist_ok=True)
training_file = training_dir / "examples.jsonl"

if training_file.exists():
    print("Keeping existing dataset:", training_file)
else:
    with training_file.open("w", encoding="utf-8") as file:
        for example in training_examples:
            file.write(json.dumps(example, ensure_ascii=False) + "\n")
    print("Saved examples:", len(training_examples))

print("Location:", training_file)


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files=str(training_file),
    split="train",
)

print("Examples:", len(dataset))
print(dataset[0]["messages"])


In [ ]:
for index, example in enumerate(dataset):
    messages = example["messages"]
    roles = [message["role"] for message in messages]

    if roles != ["system", "user", "assistant"]:
        raise ValueError(f"Example {index}: incorrect message roles.")

    for message in messages:
        if not isinstance(message["content"], str) or not message["content"].strip():
            raise ValueError(f"Example {index}: empty or invalid content.")

print(f"All {len(dataset)} examples have valid structure.")


## 5. Tokenizer and sequence lengths


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

formatted_example = tokenizer.apply_chat_template(
    dataset[0]["messages"],
    tokenize=False,
    add_generation_prompt=False,
)
print(formatted_example)


In [ ]:
def count_tokens(example):
    token_ids = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=True,
        return_dict=False,
        add_generation_prompt=False,
    )
    return {"token_count": len(token_ids)}


dataset = dataset.map(count_tokens)
print("Token counts:", dataset["token_count"])
print("Longest example:", max(dataset["token_count"]))


## 6. LoRA configuration


In [ ]:
from peft import LoraConfig, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)
print(lora_config)


## 7. Load model and attach adapter
Float32 requires substantial memory. This cell loads the model but does not start training.


In [ ]:
from transformers import AutoModelForCausalLM
from peft import get_peft_model

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float32,
)
model = get_peft_model(model, lora_config)
model.config.use_cache = False
model.to(device)
model.print_trainable_parameters()


## 8. Prepare assistant-only training labels
The full conversation is input. Labels of -100 exclude the prompt from the loss.


In [ ]:
def tokenize_example(example):
    messages = example["messages"]

    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=False,
        add_generation_prompt=False,
    )

    prompt_ids = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=True,
        return_dict=False,
        add_generation_prompt=True,
    )

    if input_ids[:len(prompt_ids)] != prompt_ids:
        raise ValueError("Prompt tokens do not match the conversation prefix.")

    labels = [-100] * len(prompt_ids) + input_ids[len(prompt_ids):]

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
    }


tokenized_dataset = dataset.map(
    tokenize_example,
    remove_columns=dataset.column_names,
)

print("Total tokens:", len(tokenized_dataset[0]["input_ids"]))
print(
    "Tokens used for learning:",
    sum(label != -100 for label in tokenized_dataset[0]["labels"]),
)


## 9. Batch padding
Preserve assistant-only labels. Padding labels must also be -100.


In [ ]:
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


def collate_examples(examples):
    features = [
        {"input_ids": example["input_ids"], "attention_mask": example["attention_mask"]}
        for example in examples
    ]
    batch = tokenizer.pad(features, padding=True, return_tensors="pt")
    batch_length = batch["input_ids"].shape[1]
    batch["labels"] = torch.tensor([
        example["labels"] + [-100] * (batch_length - len(example["labels"]))
        for example in examples
    ], dtype=torch.long)
    return batch


model_limit = model.config.max_position_embeddings
for example in tokenized_dataset:
    if len(example["input_ids"]) > model_limit:
        raise ValueError("An example exceeds the model context limit. Shorten its evidence.")
    if not any(label != -100 for label in example["labels"]):
        raise ValueError("An example contains no assistant tokens to learn from.")

batch = collate_examples([tokenized_dataset[0], tokenized_dataset[-1]])
assert torch.all(batch["labels"][batch["attention_mask"] == 0] == -100)
print("Batch shape:", batch["input_ids"].shape)


## 10. Training settings
One epoch is a workflow test. Two examples do not establish an improvement. No evaluation split is used for this practice run.


In [ ]:
from transformers import Trainer, TrainingArguments, set_seed

set_seed(42)
if device == "cpu":
    torch.set_num_threads(2)

output_dir = PROJECT_ROOT / "models" / "qwen-study-lora"
training_args = TrainingArguments(
    output_dir=str(output_dir / "checkpoints"),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    learning_rate=1e-4,
    optim="adamw_torch",
    fp16=False,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=1,
    save_strategy="no",
    eval_strategy="no",
    report_to="none",
    dataloader_pin_memory=False,
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=collate_examples,
    processing_class=tokenizer,
)


## 11. Train and save
Saves the LoRA adapter and tokenizer locally. The original Qwen weights remain separate and are required when reloading.


In [ ]:
train_result = trainer.train()

adapter_dir = output_dir / "adapter"
trainer.save_model(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
trainer.save_metrics("train", train_result.metrics)
trainer.save_state()

print("Training loss:", train_result.metrics["train_loss"])
print("Saved adapter and tokenizer:", adapter_dir)


## 12. Reload the saved adapter
Release training objects first so a second copy of Qwen does not remain in memory.


In [ ]:
import gc
from peft import PeftModel

trainer.optimizer = None
del trainer, model, batch
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
elif device == "mps":
    torch.mps.empty_cache()

saved_tokenizer = AutoTokenizer.from_pretrained(str(adapter_dir))
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.float32 if device == "cpu" else torch.float16,
)
fine_tuned_model = PeftModel.from_pretrained(base_model, str(adapter_dir))
fine_tuned_model.to(device)
fine_tuned_model.eval()
fine_tuned_model.config.use_cache = True
print("Saved adapter reloaded successfully.")


## 13. Generate an answer
This training example is only a reload check, not an independent quality test.


In [ ]:
test_messages = dataset[0]["messages"][:-1]
inputs = saved_tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
).to(device)
input_length = inputs["input_ids"].shape[1]
max_new_tokens = 100
if input_length + max_new_tokens > fine_tuned_model.config.max_position_embeddings:
    raise ValueError("Shorten the test prompt or answer length.")

with torch.inference_mode():
    generated = fine_tuned_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=saved_tokenizer.pad_token_id,
    )

answer = saved_tokenizer.decode(
    generated[0, input_length:], skip_special_tokens=True,
)
print(answer)
